# SGD Classifier & Regressor

Stochastic Gradient Descent (SGD) optimization for linear models. Efficient for very large datasets and supports online/incremental learning via `partial_fit`. Implements various loss functions for both classification and regression.

**When to Use:**
- Very large datasets (>100k samples) where batch methods are too slow
- Streaming data or online learning scenarios (incremental updates via partial_fit)
- When training speed matters more than finding the global optimum
- As a scalable alternative to Logistic Regression or Linear Regression

**Key Assumptions / Considerations:**
- Features MUST be scaled (critical for SGD convergence)
- Data is IID (independent and identically distributed)
- Linear relationship between features and target
- Learning rate and regularization require careful tuning
- Results can vary between runs (stochastic nature) — use random_state for reproducibility

**References:**
- [sklearn SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html)
- [sklearn SGDRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html)
- [sklearn SGD Tips](https://scikit-learn.org/stable/modules/sgd.html#tips-on-practical-use)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier, SGDRegressor
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

# numeric
age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

# categorical
gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

# continuous target
target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)

# binary classification target
target_binary = (target_continuous > np.median(target_continuous)).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_continuous  # switch to target_binary for classification 
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Distribution

plt.figure(figsize=(8, 5))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

print("Skewness:", round(y.skew(),4))
print("Kurtosis:", round(y.kurt(), 4))

In [ ]:
# Train/Test Split

try:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "SGD_Regressor_L2": (SGDRegressor(random_state=42, max_iter=1000, tol=1e-3), {
        "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__loss": ["squared_error", "huber", "epsilon_insensitive"],
        "clf__penalty": ["l2"],
    }),
    "SGD_Regressor_ElasticNet": (SGDRegressor(random_state=42, max_iter=1000, tol=1e-3), {
        "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__l1_ratio": [0.15, 0.5, 0.85],
        "clf__penalty": ["elasticnet"],
    }),
    "SGD_Classifier_Log": (SGDClassifier(random_state=42, max_iter=1000, tol=1e-3), {
        "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__loss": ["log_loss", "modified_huber"],
        "clf__penalty": ["l2", "l1", "elasticnet"],
    }),
    "SGD_Classifier_Hinge": (SGDClassifier(random_state=42, max_iter=1000, tol=1e-3, loss="hinge"), {
        "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__penalty": ["l2", "l1", "elasticnet"],
    }),
}

# Cross Validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1) if len(y.unique()) < 10 else KFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n\U0001f539 Training {name}...")

        is_classifier = isinstance(model, SGDClassifier)

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        scoring = "roc_auc" if is_classifier else "r2"

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False)

        try:
            y_train_use = y_train if not is_classifier else (y_train > np.median(y_train)).astype(int)
            y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)

            grid.fit(X_train, y_train_use)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            y_pred = best_model.predict(X_test)

            if is_classifier:
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "Accuracy": accuracy_score(y_test_use, y_pred),
                    "Recall": recall_score(y_test_use, y_pred),
                    "Precision": precision_score(y_test_use, y_pred),
                    "F1 Score": f1_score(y_test_use, y_pred),
                }
                # ROC-AUC only for models with predict_proba or decision_function
                if hasattr(best_model, "predict_proba"):
                    metrics["ROC-AUC"] = roc_auc_score(y_test_use, best_model.predict_proba(X_test)[:, 1])
                elif hasattr(best_model.named_steps["clf"], "decision_function"):
                    metrics["ROC-AUC"] = roc_auc_score(y_test_use, best_model.decision_function(X_test))
            else:
                n = len(y_test_use)
                p = X_train.shape[1]
                rss = np.sum((y_test_use - y_pred) ** 2)
                mse = mean_squared_error(y_test_use, y_pred)
                aic = n * np.log(rss / n) + 2 * p
                bic = n * np.log(rss / n) + p * np.log(n)
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "RMSE": np.sqrt(mse),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R\u00b2 Score": r2_score(y_test_use, y_pred),
                    "AIC": aic,
                    "BIC": bic,
                }

            results.append(metrics)
        except Exception as e:
            print(f"\u26a0\ufe0f Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n\u2705 Model Evaluation Summary:")
results_df

In [ ]:
# Best Model
regression_results = [r for r in results if 'R\u00b2 Score' in r]
classification_results = [r for r in results if 'F1 Score' in r]

if regression_results:
    best_reg_name = max(regression_results, key=lambda x: x['R\u00b2 Score'])['Model']
    best_reg_pipeline = [p for n, p in pipelines if n == best_reg_name][0]
    print(f"\n\U0001f3c6 Best Regression Model: {best_reg_name}")

if classification_results:
    best_clf_name = max(classification_results, key=lambda x: x['F1 Score'])['Model']
    best_clf_pipeline = [p for n, p in pipelines if n == best_clf_name][0]
    print(f"\U0001f3c6 Best Classification Model: {best_clf_name}")

In [ ]:
# Coefficients

for name, pipe in pipelines:
    print(f"\nModel: {name}")
    
    clf = pipe.named_steps["clf"]
    coef = clf.coef_.flatten()
    intercept = clf.intercept_[0] if hasattr(clf.intercept_, "__len__") else clf.intercept_
    
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    df_coef = pd.DataFrame({
        "feature": feature_names,
        "coef": coef
    })
    
    df_intercept = pd.DataFrame({
        "feature": ["Intercept"],
        "coef": [intercept]
    })
    
    df_coef = pd.concat([df_intercept, df_coef], ignore_index=True)
    print(df_coef)

In [ ]:
# Convergence Curve (Online Learning with partial_fit)

X_transformed = preprocessor.fit_transform(X_train_full)
feature_names_out = preprocessor.get_feature_names_out()

# Regression convergence
sgd_reg = SGDRegressor(random_state=42, learning_rate="optimal", eta0=0.01, max_iter=1, warm_start=False, tol=None)
reg_losses = []
n_epochs = 50

for epoch in range(n_epochs):
    # Shuffle each epoch
    idx = np.random.permutation(len(X_transformed))
    X_shuffled = X_transformed[idx]
    y_shuffled = y_train_full.values[idx]
    
    batch_size = 256
    for start in range(0, len(X_shuffled), batch_size):
        end = start + batch_size
        sgd_reg.partial_fit(X_shuffled[start:end], y_shuffled[start:end])
    
    y_pred = sgd_reg.predict(X_transformed)
    reg_losses.append(mean_squared_error(y_train_full, y_pred))

# Classification convergence
y_train_binary = (y_train_full > np.median(y_train_full)).astype(int)
sgd_clf = SGDClassifier(random_state=42, loss="log_loss", learning_rate="optimal", max_iter=1, warm_start=False, tol=None)
clf_losses = []

for epoch in range(n_epochs):
    idx = np.random.permutation(len(X_transformed))
    X_shuffled = X_transformed[idx]
    y_shuffled = y_train_binary.values[idx]
    
    for start in range(0, len(X_shuffled), batch_size):
        end = start + batch_size
        sgd_clf.partial_fit(X_shuffled[start:end], y_shuffled[start:end], classes=[0, 1])
    
    y_pred = sgd_clf.predict(X_transformed)
    clf_losses.append(1 - accuracy_score(y_train_binary, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(n_epochs), reg_losses)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].set_title("SGD Regressor - Convergence")

axes[1].plot(range(n_epochs), clf_losses)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Error Rate")
axes[1].set_title("SGD Classifier - Convergence")

plt.tight_layout()
plt.show()

print("\U0001f4a1 partial_fit enables online/incremental learning on streaming data")

In [ ]:
# Regression Diagnostics

def regression_diagnostics(model, X, y, name="Model"):
    y_pred = model.predict(X)
    residuals = y - y_pred
    fitted = y_pred

    plt.figure(figsize=(12, 10))
    
    plt.subplot(2,2,1)
    sns.scatterplot(x=fitted, y=residuals)
    plt.axhline(0, color='r', linestyle='--')
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{name} - Residuals vs Fitted")
    
    plt.subplot(2,2,2)
    sns.histplot(residuals, kde=True, bins=30)
    plt.title(f"{name} - Residual Distribution")
    
    plt.subplot(2,2,3)
    sns.scatterplot(x=fitted, y=np.sqrt(np.abs(residuals)))
    plt.xlabel("Fitted values")
    plt.ylabel("Sqrt(|Residuals|)")
    plt.title(f"{name} - Scale-Location")
    
    plt.subplot(2,2,4)
    sns.scatterplot(x=y, y=fitted)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{name} - Predicted vs Actual")
    
    plt.tight_layout()
    plt.show()

for name, pipe in pipelines:
    if "Regressor" in name:
        regression_diagnostics(pipe, X_test, y_test, f"{name} (Test)")
        break  # show best regression model only

In [ ]:
# Classification Diagnostics

y_test_binary = (y_test > np.median(y_test)).astype(int)

for name, pipe in pipelines:
    if "Classifier" in name:
        y_pred = pipe.predict(X_test)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        cm = confusion_matrix(y_test_binary, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")
        axes[0].set_title(f"{name} - Confusion Matrix")
        
        # ROC curve if available
        if hasattr(pipe, "predict_proba"):
            y_proba = pipe.predict_proba(X_test)[:, 1]
        elif hasattr(pipe.named_steps["clf"], "decision_function"):
            y_proba = pipe.decision_function(X_test)
        else:
            y_proba = None
        
        if y_proba is not None:
            fpr, tpr, _ = roc_curve(y_test_binary, y_proba)
            auc = roc_auc_score(y_test_binary, y_proba)
            axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
            axes[1].plot([0, 1], [0, 1], 'r--')
            axes[1].set_xlabel("False Positive Rate")
            axes[1].set_ylabel("True Positive Rate")
            axes[1].set_title(f"{name} - ROC Curve")
            axes[1].legend()
        else:
            axes[1].text(0.5, 0.5, "No probability output\n(hinge loss)", ha='center', va='center')
            axes[1].set_title(f"{name} - ROC Curve N/A")
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{name}:")
        print(classification_report(y_test_binary, y_pred))

In [ ]:
# Profile Plots

def plot_profiles(best_pipeline, X, y_true):
    y_pred = best_pipeline.predict(X)
    
    n_cols = 3
    n_features = X.shape[1]
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()  

    for i, col in enumerate(X.columns):
        temp = pd.DataFrame({
            col: X[col],
            "y_true": y_true,
            "y_pred": y_pred
        })

        grouped = temp.groupby(col).agg(
            count=("y_true", "size"),
            avg_true=("y_true", "mean"),
            avg_pred=("y_pred", "mean")
        ).reset_index().sort_values(col)

        ax1 = axes[i]
        ax1.bar(grouped[col], grouped["count"], alpha=0.3)
        ax1.set_xlabel(col)
        ax1.set_ylabel("Population")

        ax2 = ax1.twinx()
        ax2.plot(grouped[col], grouped["avg_true"], marker="o", label="Actual Target")
        ax2.plot(grouped[col], grouped["avg_pred"], marker="o", linestyle="--", label="Predicted Target")
        ax2.set_ylabel("Target Value")

        ax1.set_title(col)
        ax2.legend(loc="upper right")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# Use best regression model for profile plots
if regression_results:
    plot_profiles(best_reg_pipeline, X_train_full, y_train_full)

In [ ]:
# future work:
#   learning rate schedules: compare "constant", "optimal", "invscaling", "adaptive"
#   warm_start=True for continued training across data batches
#   averaged SGD (average=True) for better generalization
#   compare with sklearn.linear_model.PassiveAggressiveClassifier/Regressor
#   kernel approximation with Nystroem or RBFSampler for non-linear SGD